# NN3 라운드 1 — 조각 0 (네 조각을 런타임 하나씩)

계획서 3A단계 1라운드입니다. 출발 모델이 직접 운전하며 고를 수 있는 순간의 25%에서 허용 녹색을 모두 같은 미래로 굴려 비교하고, 학생이 '확실한 곳만' 바꾸도록 배웁니다. 이 부분이 강화학습입니다.
방법과 읽는 법은 `results/explore/nn3_20261103/사전등록.md`에 고정했습니다.

`NN3_P_prep`이 끝난 뒤 `shard0`~`shard3`을 **각각 런타임 하나씩** 씁니다. 순서는 상관없고 동시에 돌리면 됩니다. 넷을 다 돌려야 다음 단계로 넘어갑니다.

1. 런타임 유형 **CPU + 높은 RAM(CPU 8개)** 을 고르고 **런타임 → 모두 실행**을 누르세요.
2. 이 노트북 하나가 끝까지 알아서 돕니다: 자료 15판(약 8시간) → 다른 조각 기다림 → 학생 학습 → 거르기·문턱 → 확인(약 2시간). 합쳐 약 10시간(어림). 런타임 사이는 Drive의 상태 파일로 맞춥니다.
3. 끊기면 그 노트북만 다시 **모두 실행**하세요. 끝난 부분은 건너뜁니다. 다른 조각은 기다립니다.
4. 네 조각이 모두 끝나면 `NN3_P_R1_judge`를 실행하고 "끝났어"라고 알려 주세요. 결과: Drive `FutureTraffic/explore_nn3/r1/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_NN3.zip"
WORKSPACE = Path("/content/FutureTraffic_NN3")
NN1 = DRIVE / "explore_nn1" / "P" / "aug"                     # N5 모델·기록(읽기만, 해시 대조)
TEACHER = DRIVE / "explore_platoon1" / "students" / "seed_42"  # 선생님 t20의 학생(읽기만, 해시 대조)
PREP = DRIVE / "explore_nn3" / "prep"
MIN_CPUS = 8
for path in (ARCHIVE, NN1 / "models" / "r1" / "seed_42" / "model.pt", TEACHER):
    assert path.exists(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", DRIVE / "explore_nn3")
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "NN3_V1", "NN3 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("nn3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON0 설정 {package_manifest['platoon0_config_sha256'][:12]} | 측정 계획 {package_manifest['plan_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_nn3.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 자료 → 학생 → 거르기 → 문턱 → 확인. 백그라운드로 돌아 셀을 멈춰도 계속 돕니다. 끊기면 처음부터 다시 모두 실행하세요.
SHARD = 0
OUT = DRIVE / "explore_nn3" / "r1"
LOGS = OUT / "logs" / f"shard{SHARD}"
assert (PREP / "start.json").exists(), "NN3_P_prep을 먼저 끝내세요."

def describe(name):
    status = OUT / "data" / "status" / f"collect_shard{SHARD}.json"
    s = colab.read_json(status) or {}
    stages = [x for x in ("data", "screen", "thresholds", "final") if (OUT / "status" / f"{x}_shard{SHARD}.json").exists()]
    return f"자료 {s.get('done', 0)}/{s.get('total', 15)}판, 실패 {len(s.get('failed', []))}, 끝난 단계 {stages}"

pids = colab.launch({f"nn3_r1_shard{SHARD}": ["src/explore_nn3.py", "round", "--prep", str(PREP), "--out", str(OUT), "--work", "/content/n3r", "--shard", str(SHARD), "--shards", "4", "--workers", str(CPUS)]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, "오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 부분은 건너뜁니다)."
print("이 조각 끝. 네 조각이 모두 끝나면 NN3_P_R1_judge를 실행하세요.")